In [1]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 6.9 MB/s eta 0:00:00


In [2]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('1_LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('1_LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [11]:
import os
from pyngrok import ngrok

In [12]:
ngrok.kill()

In [13]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://sporoid-nonnegligibly-casie.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://sporoid-nonnegligibly-casie.ngrok-free.dev


True

In [14]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch()  #校長是最新的是因為使用了google search的工具
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [15]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [16]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Minghsin University of Science and Technology，簡稱明新科大）是一所位於台灣新竹縣新豐鄉的私立科技大學。學校的創立宗旨源於《大學》中「在明明德，在新民，在止於至善」的精義，並以「堅毅、求新、創造」為校訓，致力於培育具備高尚品德、專業學問與優良技術的科技人才。

明新科技大學的歷史可追溯至1966年3月成立的「明新工業專科學校」。歷經多年的發展與改制，於1997年升格為「明新技術學院」，並最終在2002年9月奉教育部核准，正式改名為「明新科技大學」。

學校目前設有六大學院，包括半導體學院、工程學院、管理學院、民生學院、人文與設計學院以及共同教育學院。明新科大積極配合國家產業發展趨勢，近年來特別鎖定半導體、AI、元宇宙、風電綠能等前瞻產業，並發展出「多元學習、全球視野、永續經營與技術創新」四大育才特色。學校斥資打造「半導體產業設備廠務與檢測人才培育基地」，並設有半導體科技博士學位學程，顯示其在半導體人才培育上的深耕與投入。

此外，明新科大也致力於推動國際化，擁有全國名列前茅的國際學生人數，校園內學生來自超過16個國家地區，營造出多元的學習環境。學校的願景目標為「深耕在地、放眼國際」，並以培育具實務經驗與人文素養之專業人才為目標。憑藉其產學合作的深度與廣度，明新科大已成為企業界青睞的人才培育基地，畢業生在業界享有良好的聲譽.


In [17]:
result2 = stateful_query("校長是誰？")
print(result2)

明新科技大學的現任校長是呂明峯教授。他於2025年2月1日正式上任，擔任明新科技大學的第11任校長。


In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [22/May/2026 03:25:43] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U3169a962e22ea2afc45f6eb98941b9c1","events":[]}
BODY:  {"destination":"U3169a962e22ea2afc45f6eb98941b9c1","events":[{"type":"message","message":{"type":"text","id":"615034369102315964","quoteToken":"uIY9LyDhYBtFqhwnsvv2M7HJOS7_ZqgujCeQqLYWMajtjNPgZsgjYQJiPVJfVeLheAmck8vzlL0rL50BjQ-mV0qqkZjNWeTaOwz-BDSO49hz-2mT6cZVh92LpSzpCi3ObjpG1vCzPe0mc-7ju7jGXQ","markAsReadToken":"Z1nEF0vMN3_Iq5ZZotRdNY0Ves_q4anyGMwWzzBNo0nGGCP_RA8igc-ccG_agI5SkZPeZ3EpFYYAdFbl2SoI3Atfhkdd29vXY5Izsv89dZm9M8xecovb19GTnhHry3vxSJ2ZqcOGrdav48OuXKcuF_L81lzG7TVeQ_kY9_r4wsEdxlCOlOFYb2UEDEcLlDHX4yJE9SPZJl8UvJldvP3HKg","text":"AI 介紹明新科大,20字以內"},"webhookEventId":"01KS6VFPXHBZZNC3R9WVWV8JRW","deliveryContext":{"isRedelivery":false},"timestamp":1779420355070,"source":{"type":"user","userId":"Uc736f30f5abec3489b5e495c080336e4"},"replyToken":"8fe1aa96b9714fbba848a69fdfefae6e","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 03:25:58] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U3169a962e22ea2afc45f6eb98941b9c1","events":[{"type":"message","message":{"type":"text","id":"615034384336552448","quoteToken":"hF2eGzvKWy6XVpbDk_WpQ9karJg3x4ssOX3KhtNRQoq_QI22t8sbEnjPukD0s8fmzaCuVHf0T5EmlnkRkt3QtX3p4B4PYjSvqMEnfHJNXs6uGwWqH-CUL2EJUikY_BhWh1qQCcOUFG4h6cukDPqx_g","markAsReadToken":"u1wl94iuT79YFsFyV0sCguA-FiosSA4U-LiLo7Z6x-BPKwfYvqusYZmlCpf9oUypNEFDamq_QzSVUfN0DHYbSIe98Iiei5tAAb59PtsoOfSFGVol9czcC2O_sAw54GrEL5DcibO65Z_FxAdY24wRwo4T2GA9zExkQvGvtn7EaVEdRN4agCndrD7q69DanYLsJ93_yoizG2mbg6UtOmbrLw","text":"AI 校長是誰"},"webhookEventId":"01KS6VFZM9BSWR6ASB375FMMMN","deliveryContext":{"isRedelivery":false},"timestamp":1779420364055,"source":{"type":"user","userId":"Uc736f30f5abec3489b5e495c080336e4"},"replyToken":"2758cc7e95274d23bbc0609b1d8273f7","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 03:26:06] "POST / HTTP/1.1" 200 -
